# Platform Inference Entry

Use this notebook only for platform py generation. Select the single code cell below. It is self-contained and does not import handler.py.


In [ ]:
"""Self-contained platform entry cell for single-image weather prediction.

Use this notebook only for platform py generation. Select this single code cell.
"""
from pathlib import Path
from typing import Any, Dict
import sys

import torch
from PIL import Image


def _find_project_root() -> Path:
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())
    names = (".", "weather_competition", "weather_competition-1")
    for start in starts:
        for name in names:
            candidate = (start / name).resolve() if name != "." else start
            if (candidate / "src").exists() and (candidate / "config.py").exists():
                return candidate
    return starts[0]


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config as cfg
from src.dataset import build_transforms
from src.model import build_model
from src.utils import get_device, load_checkpoint, load_checkpoint_payload, load_json

_MODEL = None
_TRANSFORM = None
_IDX_TO_CLASS = None
_DEVICE = None


def _resolve_path(path_value) -> Path:
    path = Path(path_value)
    if path.exists():
        return path
    candidate = PROJECT_ROOT / path
    if candidate.exists():
        return candidate
    return path


def _load_runtime():
    global _MODEL, _TRANSFORM, _IDX_TO_CLASS, _DEVICE
    if _MODEL is not None:
        return _MODEL, _TRANSFORM, _IDX_TO_CLASS, _DEVICE

    _DEVICE = get_device()
    idx_path = _resolve_path(cfg.IDX_TO_CLASS_PATH)
    model_path = _resolve_path(cfg.BEST_MODEL_PATH)
    _IDX_TO_CLASS = load_json(idx_path)
    num_classes = len(_IDX_TO_CLASS)

    checkpoint = load_checkpoint_payload(model_path, _DEVICE)
    model_name = checkpoint.get("model_name", cfg.MODEL_NAME)
    _MODEL, _ = build_model(model_name, num_classes, pretrained=False, fallback_model_name=cfg.FALLBACK_MODEL_NAME)
    _MODEL = _MODEL.to(_DEVICE)
    load_checkpoint(_MODEL, model_path, _DEVICE)
    _MODEL.eval()
    _TRANSFORM = build_transforms(cfg.IMG_SIZE, is_train=False, mean=cfg.MEAN, std=cfg.STD)
    return _MODEL, _TRANSFORM, _IDX_TO_CLASS, _DEVICE


def _read_image(data: Any) -> Image.Image:
    if isinstance(data, dict):
        data = data.get("image") or data.get("image_path") or data.get("path")
    if isinstance(data, Image.Image):
        return data.convert("RGB")
    if isinstance(data, (str, Path)):
        return Image.open(data).convert("RGB")
    raise ValueError("Unsupported input. Expected image path, PIL image, or dict with image path.")


def handle(data: Any) -> Dict[str, Any]:
    try:
        model, transform, idx_to_class, device = _load_runtime()
        image = _read_image(data)
        tensor = transform(image).unsqueeze(0).to(device)
        with torch.inference_mode():
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1)
            confidence, pred_idx = probs.max(dim=1)
        label = idx_to_class.get(str(int(pred_idx.item())), str(int(pred_idx.item())))
        return {"label": label, "confidence": float(confidence.item())}
    except Exception as exc:
        return {"error": str(exc)}


def predict(data: Any) -> str:
    result = handle(data)
    if isinstance(result, dict):
        label = result.get("label")
        if isinstance(label, str) and label:
            return label
    return "cloudy"


